# YOLO Training Monitor

Paste a training run directory below (e.g. `runs/detect/train`) and run all cells to see settings, metrics, and sample images.

Re-run cells while training is in progress to refresh.

In [ ]:
# --- Config ---
RUN_DIR = "runs/detect/train"  # <-- paste your run directory here

from pathlib import Path
run_dir = Path(RUN_DIR).resolve()
assert run_dir.exists(), f"Run dir not found: {run_dir}"
print(f"Monitoring: {run_dir}")

## Training settings (`args.yaml`)

In [ ]:
import yaml

args_path = run_dir / "args.yaml"
if args_path.exists():
    with open(args_path) as f:
        args = yaml.safe_load(f)
    # Show the most relevant keys first
    key_order = ["model", "data", "epochs", "batch", "imgsz", "device", "optimizer", "lr0", "lrf", "momentum", "weight_decay", "patience", "save_dir"]
    for k in key_order:
        if k in args:
            print(f"  {k:20s} = {args[k]}")
    print("\n--- Full args ---")
    for k, v in sorted(args.items()):
        if k not in key_order:
            print(f"  {k:20s} = {v}")
else:
    print(f"No args.yaml found at {args_path}")

## Progress & metrics (`results.csv`)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_path = run_dir / "results.csv"
if not results_path.exists():
    print(f"No results.csv yet at {results_path} — training may not have written any epochs.")
else:
    df = pd.read_csv(results_path)
    df.columns = [c.strip() for c in df.columns]
    total_epochs = args.get("epochs", "?") if args_path.exists() else "?"
    current_epoch = int(df["epoch"].iloc[-1]) if "epoch" in df.columns else len(df)
    print(f"Progress: epoch {current_epoch} / {total_epochs}")
    print(f"Rows in results.csv: {len(df)}")
    display(df.tail(10))

In [ ]:
# Plot losses and mAP curves
if results_path.exists():
    loss_cols = [c for c in df.columns if "loss" in c.lower()]
    map_cols = [c for c in df.columns if "mAP" in c]
    pr_cols = [c for c in df.columns if c.split("/")[-1] in ("precision", "recall") or c.endswith("precision") or c.endswith("recall")]

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    for c in loss_cols:
        axes[0].plot(df["epoch"], df[c], label=c)
    axes[0].set_title("Losses")
    axes[0].set_xlabel("epoch")
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)

    for c in map_cols:
        axes[1].plot(df["epoch"], df[c], label=c)
    axes[1].set_title("mAP")
    axes[1].set_xlabel("epoch")
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)

    for c in pr_cols:
        axes[2].plot(df["epoch"], df[c], label=c)
    axes[2].set_title("Precision / Recall")
    axes[2].set_xlabel("epoch")
    axes[2].legend(fontsize=8)
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

## Results plot (`results.png`)

In [ ]:
from IPython.display import Image, display

results_png = run_dir / "results.png"
if results_png.exists():
    display(Image(filename=str(results_png)))
else:
    print("No results.png yet.")

## Validation samples (labels vs predictions)

In [ ]:
val_labels = sorted(run_dir.glob("val_batch*_labels.jpg"))
val_preds = sorted(run_dir.glob("val_batch*_pred.jpg"))
for lbl, pred in zip(val_labels, val_preds):
    print(f"{lbl.name}  vs  {pred.name}")
    display(Image(filename=str(lbl)))
    display(Image(filename=str(pred)))

## Weights

In [ ]:
import matplotlib.image as mpimg
import numpy as np

plot_names = [
    "confusion_matrix.png", "confusion_matrix_normalized.png",
    "BoxPR_curve.png", "BoxF1_curve.png",
    "BoxP_curve.png", "BoxR_curve.png",
    "labels.jpg",
]
existing = [(n, run_dir / n) for n in plot_names if (run_dir / n).exists()]

if existing:
    ncols = 3
    nrows = (len(existing) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, (name, p) in zip(axes, existing):
        ax.imshow(mpimg.imread(p))
        ax.set_title(name, fontsize=9)
        ax.axis("off")
    for ax in axes[len(existing):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No diagnostic plots yet.")

## Count evaluation (MSE / MAE)

Runs the trained model on the validation set and compares predicted box count vs. ground-truth count per image.
This is the metric you care about if "how many people are in this frame" is the downstream question.

In [ ]:
import matplotlib.image as mpimg
import numpy as np

plot_names = [
    "confusion_matrix.png", "confusion_matrix_normalized.png",
    "BoxPR_curve.png", "BoxF1_curve.png",
    "BoxP_curve.png", "BoxR_curve.png",
    "labels.jpg",
]
existing = [(n, run_dir / n) for n in plot_names if (run_dir / n).exists()]

if existing:
    ncols = 2
    nrows = (len(existing) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(12 * ncols, 10 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, (name, p) in zip(axes, existing):
        ax.imshow(mpimg.imread(p))
        ax.set_title(name, fontsize=14)
        ax.axis("off")
    for ax in axes[len(existing):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No diagnostic plots yet.")

In [ ]:
# 3. Load model and run inference on val set
from ultralytics import YOLO

model = YOLO(str(weights_path))

IMG_EXT = {".jpg", ".jpeg", ".png"}
val_images = sorted(p for p in val_images_dir.iterdir() if p.suffix.lower() in IMG_EXT)

pred_counts, gt_counts, names = [], [], []

results = model.predict(
    source=[str(p) for p in val_images],
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    verbose=False,
    stream=True,
)

for img_path, res in zip(val_images, results):
    # Predicted count
    pred_n = 0 if res.boxes is None else len(res.boxes)
    # Ground-truth count from label file
    lbl_path = val_labels_dir / f"{img_path.stem}.txt"
    gt_n = 0
    if lbl_path.exists():
        gt_n = sum(1 for line in lbl_path.read_text().splitlines() if line.strip())
    pred_counts.append(pred_n)
    gt_counts.append(gt_n)
    names.append(img_path.name)

pred_counts = np.array(pred_counts)
gt_counts = np.array(gt_counts)
errors = pred_counts - gt_counts

mae = np.mean(np.abs(errors))
mse = np.mean(errors ** 2)
rmse = np.sqrt(mse)
bias = np.mean(errors)

print(f"\n=== Count metrics on {len(val_images)} val images ===")
print(f"  MAE   (mean abs error)  = {mae:.3f}")
print(f"  MSE   (mean sq  error)  = {mse:.3f}")
print(f"  RMSE                    = {rmse:.3f}")
print(f"  Bias  (mean pred - gt)  = {bias:+.3f}   (positive = over-counts)")
print(f"  GT   range / mean       = [{gt_counts.min()}, {gt_counts.max()}] / {gt_counts.mean():.2f}")
print(f"  Pred range / mean       = [{pred_counts.min()}, {pred_counts.max()}] / {pred_counts.mean():.2f}")

In [ ]:
# 4. Plot predicted vs. ground-truth count + error distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lo = 0
hi = max(pred_counts.max(), gt_counts.max()) + 1
axes[0].scatter(gt_counts, pred_counts, alpha=0.6)
axes[0].plot([lo, hi], [lo, hi], "k--", alpha=0.4, label="y = x")
axes[0].set_xlabel("Ground-truth count")
axes[0].set_ylabel("Predicted count")
axes[0].set_title(f"Pred vs GT (MAE={mae:.2f}, RMSE={rmse:.2f})")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(errors, bins=range(int(errors.min()) - 1, int(errors.max()) + 2), edgecolor="k")
axes[1].axvline(0, color="k", linestyle="--", alpha=0.4)
axes[1].set_xlabel("Predicted − Ground-truth")
axes[1].set_ylabel("Number of images")
axes[1].set_title("Count error distribution")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Worst cases (largest abs error)
errs_df = pd.DataFrame({"image": names, "gt": gt_counts, "pred": pred_counts, "error": errors})
errs_df["abs_error"] = errs_df["error"].abs()
print("\nWorst 10 predictions by |error|:")
display(errs_df.sort_values("abs_error", ascending=False).head(10).reset_index(drop=True))